# Export Gemma 4 LoRA to GGUF

Merges LoRA adapters with base model and converts to Q8_0 GGUF using llama.cpp directly (bypasses Unsloth GGUF export bug).

**Runtime**: A100 GPU + High RAM. Run all cells in order.

In [ ]:
# Build llama.cpp from source
!git clone --recursive https://github.com/ggerganov/llama.cpp /root/llama.cpp
!cd /root/llama.cpp && cmake -B build && cmake --build build -j --config Release
!ls /root/llama.cpp/build/bin/llama-quantize && echo "Build OK"

In [ ]:
# Merge LoRA into base model using PEFT (no Unsloth needed)
!pip install -q peft transformers accelerate huggingface_hub sentencepiece protobuf

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading base model in FP16 (uses RAM, not GPU)...")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-4-31B-it",
    torch_dtype=torch.float16,
    device_map="cpu",
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-31B-it", token=HF_TOKEN)

print("Loading and merging LoRA...")
model = PeftModel.from_pretrained(model, "daresearch/sp500-exec-classifier-gemma4-lora")
model = model.merge_and_unload()

print("Saving merged model...")
model.save_pretrained("/tmp/gemma4_merged")
tokenizer.save_pretrained("/tmp/gemma4_merged")
print("Saved to /tmp/gemma4_merged")

In [ ]:
# Convert to Q8_0 GGUF using llama.cpp
!python /root/llama.cpp/convert_hf_to_gguf.py /tmp/gemma4_merged --outtype q8_0 --outfile /tmp/gemma4-Q8_0.gguf
!ls -lh /tmp/gemma4-Q8_0.gguf

In [ ]:
# Upload to HuggingFace
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="/tmp/gemma4-Q8_0.gguf",
    path_in_repo="gemma-4-31B-it-Q8_0.gguf",
    repo_id="daresearch/sp500-exec-classifier-gemma4-gguf",
    token=HF_TOKEN,
)
print("Done! Uploaded to https://huggingface.co/daresearch/sp500-exec-classifier-gemma4-gguf")